# Part 10: Encoders & Sequence-to-Sequence — Beyond Decoder-Only

So far this series has been **decoder-only / GPT-focused**: causal attention, autoregressive
generation, weight-tied LM heads. That is the dominant architecture for modern general-purpose
LLMs — but it is not the *only* one, and it is not always the *right* one.

In this notebook we fill the gap with the two other members of the Transformer family:

1. **Encoder-only (BERT-style)** — *bidirectional* attention for **understanding**: classification,
   embeddings, retrieval.
2. **Encoder-decoder (the original 2017 Transformer / T5)** — an encoder that reads a source
   sequence plus a decoder that generates a target while **cross-attending** to the encoder, for
   **conditional generation**: translation, summarization.

We will reuse the library from `../src` (`TransformerEncoder`, `MultiHeadAttention`,
`TransformerEmbedding`) and inline the new pieces (a cross-attention decoder block, a tiny
seq2seq model, and a masked-language-modeling demo). Everything is small and runs on CPU in
well under a couple of minutes.

---

In [ ]:
import sys
sys.path.append('..')

import math
import random

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F

from src.transformer import TransformerEncoder, TransformerBlock, FeedForward
from src.attention import MultiHeadAttention, ScaledDotProductAttention
from src.embeddings import TransformerEmbedding

%matplotlib inline

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print(f"torch {torch.__version__} | device: {device}")

## 1. The Three Families

Notebook 06 introduced this table briefly. Here we go deeper. Every modern Transformer is one of
three arrangements of the *same* building blocks (attention + FFN + residual + LayerNorm). What
differs is **who can attend to whom**.

| | **Decoder-only (GPT)** | **Encoder-only (BERT)** | **Encoder-decoder (T5 / original)** |
|---|---|---|---|
| Attention | Causal (look left only) | Bidirectional (look everywhere) | Enc: bidir. · Dec: causal **+ cross-attn** |
| Pretraining objective | Next-token prediction | Masked LM (+ NSP-ish) | Span corruption / seq2seq |
| Natural strength | **Generation** | **Understanding** | **Conditional generation** |
| Typical tasks | Chat, completion, code | Classification, NER, retrieval embeddings | Translation, summarization |
| Sees the future? | No | Yes (whole input at once) | Decoder no; encoder yes |
| Examples | GPT, Llama, Mistral | BERT, RoBERTa, embedding models | T5, BART, the 2017 "Attention Is All You Need" |

The key mechanical knob is the **attention mask**:

- **No mask** → bidirectional → every token's representation is informed by the entire sequence
  (great for *encoding* a fixed input).
- **Causal (triangular) mask** → each position only sees itself and earlier positions (required
  for *generating* left-to-right without peeking at the answer).
- **Cross-attention** → a third mode where queries come from one sequence (the decoder) and
  keys/values from another (the encoder output) — the bridge in encoder-decoder models.

Let's make each of these concrete.

---

## 2. Encoder-only (BERT-style)

### 2.1 Bidirectional self-attention vs. the causal mask

In notebook 03 we built a **causal mask**: a lower-triangular matrix that forces position *i* to
attend only to positions ≤ *i*. That is essential for a generator — otherwise the model could
cheat by looking at the token it is supposed to predict.

An **encoder has no such constraint**. It receives the *whole* input at once and its only job is
to build rich contextual representations, so every token is allowed to attend to every other
token (including ones to its right). In our library, `TransformerBlock`/`TransformerEncoder`
use bidirectional self-attention whenever `mask=None` — exactly what we want.

Let's run the encoder on a toy embedded sequence and look at the attention matrix.

In [ ]:
# Toy sequence: 1 batch, 6 tokens, small model
torch.manual_seed(0)
d_model, num_heads, num_layers, seq_len = 64, 4, 2, 6

encoder = TransformerEncoder(d_model, num_heads, num_layers, dropout=0.0).to(device)
encoder.eval()

x = torch.randn(1, seq_len, d_model, device=device)

with torch.no_grad():
    enc_out, attn_list = encoder(x, mask=None)   # mask=None => bidirectional

print("encoder output shape:", tuple(enc_out.shape))
print("num layers of attention returned:", len(attn_list))
print("attention shape per layer (batch, heads, q, k):", tuple(attn_list[0].shape))

# Average over heads for the last layer to get a (seq, seq) attention map
attn = attn_list[-1][0].mean(0).cpu().numpy()   # (seq_len, seq_len)

# A causal mask, for contrast, would zero out the upper triangle:
causal = np.tril(np.ones((seq_len, seq_len)))
print("\nFraction of attention mass ABOVE the diagonal (future tokens):",
      round(float(attn[np.triu_indices(seq_len, k=1)].sum()) / float(attn.sum()), 3))
print("(A causal/decoder model would have exactly 0.0 here.)")

In [ ]:
# Visualize: encoder attention (full) vs. what a causal mask would allow
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))

im0 = axes[0].imshow(attn, cmap='viridis', vmin=0)
axes[0].set_title("Encoder self-attention (bidirectional)\nevery token attends everywhere")
axes[0].set_xlabel("key position"); axes[0].set_ylabel("query position")
fig.colorbar(im0, ax=axes[0], fraction=0.046)

im1 = axes[1].imshow(causal, cmap='Greys', vmin=0, vmax=1)
axes[1].set_title("Causal mask (decoder) for contrast\nupper triangle is forbidden")
axes[1].set_xlabel("key position"); axes[1].set_ylabel("query position")
fig.colorbar(im1, ax=axes[1], fraction=0.046)

plt.tight_layout(); plt.show()

print("Notice the encoder map has non-zero weights in the upper triangle —")
print("it is NOT triangular. That is the whole point of an encoder.")

### 2.2 Masked Language Modeling (MLM) — BERT's pretraining objective

A causal model trains by predicting the *next* token. But a bidirectional encoder can see the
whole sequence, so "predict the next token" is trivial (it can just look at it). BERT solves this
with **Masked Language Modeling**:

> Randomly corrupt ~**15%** of the input tokens, then ask the model to reconstruct the *original*
> tokens at those positions, using the surrounding (bidirectional) context.

The classic recipe for the chosen 15%:
- **80%** → replace with a special `[MASK]` token,
- **10%** → replace with a random token,
- **10%** → leave unchanged (but still predict it).

The 10%/10% twist prevents the model from only ever learning "fill in the `[MASK]` slot" and
forces it to build good representations for *every* position. The loss is computed **only on the
masked/selected positions** — unmasked positions contribute nothing.

Below is a tiny, fully runnable MLM demo on a synthetic **character-level** corpus.

In [ ]:
# --- Tiny synthetic corpus: repeating structured "words" over a small char vocab ---
chars = list("abcdefg")
PAD, MASK = "<pad>", "<mask>"
vocab = [PAD, MASK] + chars
stoi = {c: i for i, c in enumerate(vocab)}
itos = {i: c for c, i in stoi.items()}
VOCAB = len(vocab)
PAD_ID, MASK_ID = stoi[PAD], stoi[MASK]

SEQ = 12
# Structured data: each sequence is a tiling of the pattern "abcdefg" so there is
# real bidirectional context to exploit (the neighbours predict the missing char).
PATTERN = "abcdefg"
def make_batch(bs):
    seqs = []
    for _ in range(bs):
        start = random.randint(0, len(PATTERN) - 1)
        s = (PATTERN * 4)[start:start + SEQ]
        seqs.append([stoi[c] for c in s])
    return torch.tensor(seqs, dtype=torch.long, device=device)

print("vocab:", vocab)
print("example sequence:", ''.join(itos[i] for i in make_batch(1)[0].tolist()))

In [ ]:
def mlm_mask(batch, mask_prob=0.15):
    """Apply the BERT 80/10/10 masking recipe.
    Returns (corrupted_inputs, labels) where labels = -100 on positions we do NOT score."""
    inputs = batch.clone()
    labels = torch.full_like(batch, -100)         # -100 => ignored by cross_entropy

    prob = torch.rand(batch.shape, device=device)
    selected = prob < mask_prob                    # the ~15% we will predict
    # Guarantee at least one masked position per row (tiny batches can roll all-False)
    for r in range(batch.size(0)):
        if selected[r].sum() == 0:
            selected[r, random.randrange(batch.size(1))] = True

    labels[selected] = batch[selected]             # score originals at selected spots

    # Of the selected: 80% -> [MASK], 10% -> random token, 10% -> unchanged
    r = torch.rand(batch.shape, device=device)
    to_mask = selected & (r < 0.8)
    to_rand = selected & (r >= 0.8) & (r < 0.9)
    inputs[to_mask] = MASK_ID
    rand_tokens = torch.randint(len(chars), batch.shape, device=device) + len(["<pad>", "<mask>"])
    inputs[to_rand] = rand_tokens[to_rand]
    return inputs, selected, labels


class TinyBERT(nn.Module):
    """Encoder + an MLM head. Pure encoder-only, fully bidirectional."""
    def __init__(self, vocab_size, d_model=64, num_heads=4, num_layers=2, max_len=32):
        super().__init__()
        self.embed = TransformerEmbedding(vocab_size, d_model, max_len,
                                          dropout=0.1, learnable_pos=True)
        self.encoder = TransformerEncoder(d_model, num_heads, num_layers, dropout=0.1)
        self.mlm_head = nn.Linear(d_model, vocab_size)

    def forward(self, ids):
        h = self.embed(ids)
        h, _ = self.encoder(h, mask=None)          # bidirectional
        return self.mlm_head(h)                     # (B, T, vocab)


bert = TinyBERT(VOCAB).to(device)
print("TinyBERT params:", sum(p.numel() for p in bert.parameters()))

In [ ]:
# --- Train MLM for a few hundred steps; loss only on masked positions ---
opt = torch.optim.AdamW(bert.parameters(), lr=3e-3)
bert.train()

losses = []
for step in range(400):
    batch = make_batch(64)
    inputs, selected, labels = mlm_mask(batch, mask_prob=0.15)
    logits = bert(inputs)
    loss = F.cross_entropy(logits.view(-1, VOCAB), labels.view(-1), ignore_index=-100)
    opt.zero_grad(); loss.backward(); opt.step()
    losses.append(loss.item())
    if step % 100 == 0 or step == 399:
        print(f"step {step:3d}  masked-token loss {loss.item():.3f}")

print(f"\nloss dropped {losses[0]:.3f} -> {losses[-1]:.3f}  "
      f"(random-guess loss ~ ln(vocab)= {math.log(VOCAB):.3f})")

In [ ]:
# Quick sanity check: mask one position and see if the encoder reconstructs it
bert.eval()
seq = (PATTERN * 4)[:SEQ]
ids = torch.tensor([[stoi[c] for c in seq]], device=device)
pos = 5
corrupt = ids.clone(); corrupt[0, pos] = MASK_ID

with torch.no_grad():
    pred = bert(corrupt)[0, pos].argmax().item()

shown = list(seq); shown[pos] = "_"
print("input (masked):", ''.join(shown))
print(f"true char at pos {pos}: '{seq[pos]}'   model predicts: '{itos[pred]}'   "
      f"-> {'correct' if itos[pred]==seq[pos] else 'wrong'}")

plt.figure(figsize=(6,3))
plt.plot(losses); plt.title("TinyBERT MLM loss"); plt.xlabel("step"); plt.ylabel("loss")
plt.tight_layout(); plt.show()

### 2.3 `[CLS]` / pooling for sentence-level classification

MLM gives us good *per-token* representations. For **sentence-level** tasks (sentiment, topic,
"do these two sentences entail each other?") we need a single vector for the whole sequence.
Two common ways to pool:

- **`[CLS]` token**: prepend a special token whose final-layer hidden state is used as the
  sequence summary (BERT's approach). It is trained to aggregate via attention.
- **Mean pooling**: average the token representations (popular for sentence-embedding models like
  Sentence-BERT).

Either way: `pooled = pool(encoder_outputs)` → `Linear` → class logits. Here is a small
conceptual demo using mean pooling on a 2-class toy problem (does the sequence contain more of
the first half of the alphabet or the second?).

In [ ]:
class TinyClassifier(nn.Module):
    def __init__(self, vocab_size, d_model=64, num_heads=4, num_layers=2, n_classes=2, max_len=32):
        super().__init__()
        self.embed = TransformerEmbedding(vocab_size, d_model, max_len, dropout=0.1)
        self.encoder = TransformerEncoder(d_model, num_heads, num_layers, dropout=0.1)
        self.cls = nn.Linear(d_model, n_classes)

    def forward(self, ids):
        h = self.embed(ids)
        h, _ = self.encoder(h, mask=None)
        pooled = h.mean(dim=1)            # mean-pool over the sequence -> (B, d_model)
        return self.cls(pooled)


def make_cls_batch(bs):
    """Label 1 if sequence is majority 'late' alphabet (e,f,g), else 0."""
    X, y = [], []
    for _ in range(bs):
        late = random.random() < 0.5
        pool = "efg" if late else "abc"
        s = [stoi[random.choice(pool if random.random()<0.8 else chars)] for _ in range(SEQ)]
        X.append(s); y.append(1 if late else 0)
    return (torch.tensor(X, device=device), torch.tensor(y, device=device))


clf = TinyClassifier(VOCAB).to(device)
opt = torch.optim.AdamW(clf.parameters(), lr=3e-3)
clf.train()
for step in range(200):
    X, y = make_cls_batch(64)
    loss = F.cross_entropy(clf(X), y)
    opt.zero_grad(); loss.backward(); opt.step()

clf.eval()
with torch.no_grad():
    Xt, yt = make_cls_batch(500)
    acc = (clf(Xt).argmax(-1) == yt).float().mean().item()
print(f"held-out sentence-classification accuracy: {acc:.1%}  (chance = 50%)")

## 3. Encoder-decoder (the original Transformer)

The 2017 "Attention Is All You Need" model was an **encoder-decoder**, designed for translation.
It has three attention flavors:

1. **Encoder self-attention** — bidirectional over the *source* (we already have this:
   `TransformerEncoder`).
2. **Decoder masked self-attention** — *causal* over the *target generated so far* (so it can't
   peek ahead, just like GPT).
3. **Cross-attention** — the new ingredient: the decoder's **queries** attend to the encoder's
   output as **keys and values**.

### 3.1 Cross-attention

```
Decoder hidden state  ->  Q  (what the decoder is currently asking about)
Encoder output        ->  K, V  (what the source sequence contains / provides)
```

This is the bridge that lets the decoder *condition* its generation on the source. Mechanically
it is just `MultiHeadAttention` with a **different query than key/value** — and our `src`
`MultiHeadAttention` already supports exactly that signature `(query, key, value, mask)`.

We inline a `DecoderBlockWithCrossAttn`: Pre-LN, with causal self-attention, then cross-attention
to the encoder memory, then an FFN — each wrapped in a residual connection.

In [ ]:
class DecoderBlockWithCrossAttn(nn.Module):
    """One encoder-decoder *decoder* block (Pre-LN):
        x = x + SelfAttn(LN(x), causal_mask)
        x = x + CrossAttn(LN(x) -> Q ; encoder_memory -> K,V)
        x = x + FFN(LN(x))
    """
    def __init__(self, d_model, num_heads, d_ff=None, dropout=0.1):
        super().__init__()
        self.self_attn  = MultiHeadAttention(d_model, num_heads, dropout)
        self.cross_attn = MultiHeadAttention(d_model, num_heads, dropout)
        self.ffn = FeedForward(d_model, d_ff, dropout)
        self.norm1 = nn.LayerNorm(d_model)   # before self-attention
        self.norm2 = nn.LayerNorm(d_model)   # before cross-attention
        self.norm3 = nn.LayerNorm(d_model)   # before FFN
        self.drop = nn.Dropout(dropout)

    def forward(self, x, memory, causal_mask=None, mem_mask=None):
        # 1) Causal self-attention over the target so far
        h = self.norm1(x)
        sa, self_w = self.self_attn(h, h, h, causal_mask)
        x = x + self.drop(sa)
        # 2) Cross-attention: queries from decoder, keys/values from encoder memory
        h = self.norm2(x)
        ca, cross_w = self.cross_attn(h, memory, memory, mem_mask)
        x = x + self.drop(ca)
        # 3) Position-wise FFN
        h = self.norm3(x)
        x = x + self.drop(self.ffn(h))
        return x, self_w, cross_w


def causal_mask(seq_len, device):
    """(1, 1, T, T) lower-triangular mask; 1 = keep, 0 = block (matches src convention)."""
    m = torch.tril(torch.ones(seq_len, seq_len, device=device))
    return m.view(1, 1, seq_len, seq_len)

print("DecoderBlockWithCrossAttn defined.")

### 3.2 Assembling a tiny `Seq2SeqTransformer`

Now we stack it all together: a source embedding + `TransformerEncoder`, a target embedding + our
cross-attention decoder blocks, and an LM head projecting to the target vocab.

In [ ]:
class Seq2SeqTransformer(nn.Module):
    def __init__(self, vocab_size, d_model=64, num_heads=4,
                 enc_layers=2, dec_layers=2, max_len=16, dropout=0.1):
        super().__init__()
        self.src_embed = TransformerEmbedding(vocab_size, d_model, max_len, dropout)
        self.tgt_embed = TransformerEmbedding(vocab_size, d_model, max_len, dropout)
        self.encoder = TransformerEncoder(d_model, num_heads, enc_layers, dropout=dropout)
        self.dec_layers = nn.ModuleList([
            DecoderBlockWithCrossAttn(d_model, num_heads, dropout=dropout)
            for _ in range(dec_layers)
        ])
        self.dec_norm = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size)

    def encode(self, src):
        memory, _ = self.encoder(self.src_embed(src), mask=None)   # bidirectional
        return memory

    def decode(self, tgt, memory):
        h = self.tgt_embed(tgt)
        cmask = causal_mask(tgt.size(1), tgt.device)
        for layer in self.dec_layers:
            h, _, _ = layer(h, memory, causal_mask=cmask)
        return self.lm_head(self.dec_norm(h))

    def forward(self, src, tgt_in):
        memory = self.encode(src)
        return self.decode(tgt_in, memory)

print("Seq2SeqTransformer defined.")

### 3.3 Toy task: sequence reversal with teacher forcing

To make cross-attention concrete we train the model on **sequence reversal**: given a random
digit string like `3 1 4 1 5`, produce its reverse `5 1 4 1 3`. This *cannot* be solved without
genuinely consulting the source, so it exercises cross-attention directly.

**Teacher forcing**: during training the decoder is fed the *shifted* gold target — it starts with
a `<sos>` token and at each step sees the correct previous target token, predicting the next one.
We build:

- `src`        = the digits
- `tgt_in`     = `<sos>` + reversed[:-1]   (decoder input)
- `tgt_out`    = reversed + `<eos>`        (what we score against)

Vocab is tiny (digits 0–9 plus `<pad> <sos> <eos>`) and sequences are short, so it trains fast.

In [ ]:
# Vocab for the seq2seq task
DIGITS = list("0123456789")
S_PAD, S_SOS, S_EOS = "<pad>", "<sos>", "<eos>"
svocab = [S_PAD, S_SOS, S_EOS] + DIGITS
s_stoi = {c: i for i, c in enumerate(svocab)}
s_itos = {i: c for c, i in s_stoi.items()}
SVOCAB = len(svocab)
SOS_ID, EOS_ID, SPAD_ID = s_stoi[S_SOS], s_stoi[S_EOS], s_stoi[S_PAD]
SRC_LEN = 6        # digits per source string

def make_reverse_batch(bs):
    src, tgt_in, tgt_out = [], [], []
    for _ in range(bs):
        digits = [random.randrange(10) for _ in range(SRC_LEN)]
        src_ids = [s_stoi[DIGITS[d]] for d in digits]
        rev_ids = src_ids[::-1]
        src.append(src_ids)
        tgt_in.append([SOS_ID] + rev_ids)          # <sos>, r0, r1, ...
        tgt_out.append(rev_ids + [EOS_ID])         # r0, r1, ..., <eos>
    t = lambda a: torch.tensor(a, dtype=torch.long, device=device)
    return t(src), t(tgt_in), t(tgt_out)

s, ti, to = make_reverse_batch(1)
print("source :", ''.join(s_itos[i] for i in s[0].tolist()))
print("tgt_in :", ' '.join(s_itos[i] for i in ti[0].tolist()))
print("tgt_out:", ' '.join(s_itos[i] for i in to[0].tolist()))

In [ ]:
seq2seq = Seq2SeqTransformer(SVOCAB, d_model=64, num_heads=4,
                             enc_layers=2, dec_layers=2,
                             max_len=SRC_LEN + 2).to(device)
print("Seq2Seq params:", sum(p.numel() for p in seq2seq.parameters()))

# --- BEFORE training: greedy decode is garbage ---
@torch.no_grad()
def greedy_decode(model, src, max_len=SRC_LEN + 1):
    model.eval()
    memory = model.encode(src)
    ys = torch.full((src.size(0), 1), SOS_ID, dtype=torch.long, device=device)
    for _ in range(max_len):
        logits = model.decode(ys, memory)
        nxt = logits[:, -1].argmax(-1, keepdim=True)
        ys = torch.cat([ys, nxt], dim=1)
    return ys[:, 1:]   # drop the <sos>

def decode_to_str(ids):
    out = []
    for i in ids.tolist():
        if i == EOS_ID: break
        if i in (SPAD_ID, SOS_ID): continue
        out.append(s_itos[i])
    return ''.join(out)

test_src, _, _ = make_reverse_batch(1)
print("source            :", ''.join(s_itos[i] for i in test_src[0].tolist()))
print("expected (reverse):", ''.join(s_itos[i] for i in test_src[0].tolist()[::-1]))
print("prediction BEFORE :", decode_to_str(greedy_decode(seq2seq, test_src)[0]))

In [ ]:
# --- Train with teacher forcing ---
opt = torch.optim.AdamW(seq2seq.parameters(), lr=3e-3)
losses = []
seq2seq.train()
for step in range(600):
    src, tgt_in, tgt_out = make_reverse_batch(64)
    logits = seq2seq(src, tgt_in)                       # teacher forcing: feed gold shifted target
    loss = F.cross_entropy(logits.reshape(-1, SVOCAB), tgt_out.reshape(-1),
                           ignore_index=SPAD_ID)
    opt.zero_grad(); loss.backward(); opt.step()
    losses.append(loss.item())
    if step % 100 == 0 or step == 599:
        print(f"step {step:3d}  loss {loss.item():.3f}")

plt.figure(figsize=(6,3))
plt.plot(losses); plt.title("Seq2Seq (reversal) training loss")
plt.xlabel("step"); plt.ylabel("loss"); plt.tight_layout(); plt.show()

In [ ]:
# --- AFTER training: evaluate generalization on fresh held-out examples ---
seq2seq.eval()
src, _, _ = make_reverse_batch(500)
pred = greedy_decode(seq2seq, src)

correct = 0
for i in range(src.size(0)):
    expected = ''.join(s_itos[t] for t in src[i].tolist()[::-1])
    got = decode_to_str(pred[i])
    correct += (got == expected)
seq_acc = correct / src.size(0)

print(f"held-out exact-sequence reversal accuracy: {seq_acc:.1%}\n")

# Show a few concrete held-out examples
demo_src, _, _ = make_reverse_batch(5)
demo_pred = greedy_decode(seq2seq, demo_src)
print(f"{'source':<10}{'expected':<12}{'predicted':<12} ok?")
for i in range(5):
    s_ = ''.join(s_itos[t] for t in demo_src[i].tolist())
    e_ = s_[::-1]
    p_ = decode_to_str(demo_pred[i])
    print(f"{s_:<10}{e_:<12}{p_:<12} {'YES' if p_==e_ else 'no'}")

In [ ]:
# Peek at the cross-attention: where does each decoder step look in the source?
seq2seq.eval()
src, _, _ = make_reverse_batch(1)
memory = seq2seq.encode(src)
ys = greedy_decode(seq2seq, src)
ys_in = torch.cat([torch.full((1,1), SOS_ID, device=device), ys[:, :-1]], dim=1)

h = seq2seq.tgt_embed(ys_in)
cmask = causal_mask(ys_in.size(1), device)
cross_w = None
for layer in seq2seq.dec_layers:
    h, _, cross_w = layer(h, memory, causal_mask=cmask)

cw = cross_w[0].mean(0).detach().cpu().numpy()  # (tgt_len, src_len)
plt.figure(figsize=(5.5,4.5))
plt.imshow(cw, cmap='viridis')
plt.xticks(range(src.size(1)), [s_itos[i] for i in src[0].tolist()])
plt.yticks(range(ys.size(1)), [s_itos[i] for i in ys[0].tolist()])
plt.xlabel("source position (keys/values)"); plt.ylabel("decoder step (queries)")
plt.title("Cross-attention for reversal\n(expect an anti-diagonal: step k -> source end-k)")
plt.colorbar(fraction=0.046); plt.tight_layout(); plt.show()
print("The bright anti-diagonal shows the decoder learned to read the source backwards.")

## 4. When to use which

A practical decision guide:

- **Generation / open-ended text → decoder-only (GPT-style).**
  Chatbots, completion, code generation, agents. Causal attention + next-token prediction scales
  beautifully and a single model handles arbitrary instructions.

- **Understanding, classification, embeddings, retrieval → encoder-only (BERT-style).**
  When you need a *representation* of a fixed input rather than to generate from it: sentiment,
  NER, reranking, and especially **dense retrieval / semantic search embeddings**. Bidirectional
  context makes these representations richer than a left-to-right model's.

- **Conditional generation with a distinct source → encoder-decoder (T5 / original Transformer).**
  Translation, summarization, grammatical correction — anything that maps an input sequence to an
  output sequence. The encoder fully digests the source; cross-attention lets the decoder consult
  it at every step.

### The modern trend

Despite this clean taxonomy, the field has largely **consolidated onto decoder-only** for
general-purpose LLMs. A sufficiently large decoder-only model, prompted appropriately
("Translate to French: ..."), does conditional generation *and* (via in-context learning)
classification surprisingly well — without needing a separate encoder. Encoder-only models remain
dominant where you specifically want **embeddings/representations** (retrieval, RAG), and
encoder-decoders persist in specialized seq2seq pipelines, but the unification under decoder-only
is one of the defining architectural stories of modern LLMs.

---

## Summary

We stepped outside the decoder-only world that the rest of this series lives in:

- **Encoder-only (BERT)**: bidirectional self-attention (just drop the causal mask — our
  `TransformerEncoder` does this with `mask=None`), pretrained with **Masked Language Modeling**
  (15% / 80-10-10 recipe, loss only on masked positions), and pooled (`[CLS]` or mean) for
  sentence-level classification. Our TinyBERT learned to reconstruct masked characters and a
  TinyClassifier hit high accuracy from a pooled representation.
- **Encoder-decoder (original Transformer)**: we inlined a `DecoderBlockWithCrossAttn` (causal
  self-attention + **cross-attention** to encoder memory + FFN, all Pre-LN), assembled a
  `Seq2SeqTransformer`, and trained it on **sequence reversal** with **teacher forcing**. It
  generalized to held-out strings, and the cross-attention map showed a clean anti-diagonal —
  visual proof the decoder learned to read the source backwards.

### Key Takeaways

- The three families are the *same* blocks under three **attention-masking regimes**: causal
  (decoder), bidirectional (encoder), and causal-plus-**cross-attention** (encoder-decoder).
- **Cross-attention** is just multi-head attention with the **query** from one sequence and the
  **keys/values** from another — the bridge that conditions a decoder on an encoder.
- **MLM** is what makes bidirectional pretraining non-trivial: you must corrupt the input so the
  model can't trivially copy the answer it can already see.
- Match architecture to task: **generate → decoder**, **understand/embed → encoder**,
  **transform a source sequence → encoder-decoder** — while recognizing the modern gravitational
  pull toward decoder-only.

➡️ **Next:** `11_modern_architectures.ipynb` — RoPE, RMSNorm, grouped-query attention,
mixture-of-experts, and the other ingredients that turn the vanilla 2017 Transformer into a
modern LLM.